### Название скрипта: Получение погодных данных с 1975-01-01 по 2025-05-31 с сайта open-meteo.com

#### Описание:
    Скрипт для первичного сбора и обновления данных о погодных данных городов РФ
    и их сохранения в csv-файл / базу данных для дальнейшего использования в работе.

#### Основные функции:
    - Установка библиотек
    - Загрузка городов группы
    - Фильтрация городов по исполнителю
    - Получение погодных данных по городам
    - Сохранение выгруженных городов

#### Использование:
    cityes_group_1.csvCity название файлов с погодными данными городов за указанный период

#### Требования:
    - Python 3.10+
    - Зависимости: pandas, openmeteo-requests, requests-cache retry-requests numpy pandas

In [1]:
pip install openmeteo-requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install requests-cache retry-requests numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import openmeteo_requests
import requests_cache
from retry_requests import retry

In [4]:
# Для оптимизации кода выделю путm к файлу на компьютере в отдельную переменную:
pc_csv = 'D:/DA_42_Learning/Script pogody/My_code/cityes_group_1.csv'

In [5]:
# Загружаем данные
try:
    data = pd.read_csv(pc_csv)
except:
    print('Данные не загружены')

In [6]:
# Проверка первых строк файла
data.head()

,name,region,population,latitude,longitude,user
0,Москва,Москва,13 010 112,55.625578,37.606392,@ivan_kudinov48
1,Красноярск,Красноярский край,1 187 771,56.009117,92.872586,@ivan_kudinov48
2,Краснодар,Краснодарский край,1 099 344,45.035153,38.977240,@ivan_kudinov48
3,Пермь,Пермский край,1 034 002,58.014965,56.246723,@ivan_kudinov48
4,Махачкала,Дагестан,623 254,42.983024,47.504872,@ivan_kudinov48


In [7]:
# Параметры фильтрации
column_name = 'user'
filter_value = '@ivan_kudinov48'

In [8]:
# Фильтрация данных
filtered_data = data[data[column_name] == filter_value]
filtered_data.head()

,name,region,population,latitude,longitude,user
0,Москва,Москва,13 010 112,55.625578,37.606392,@ivan_kudinov48
1,Красноярск,Красноярский край,1 187 771,56.009117,92.872586,@ivan_kudinov48
2,Краснодар,Краснодарский край,1 099 344,45.035153,38.977240,@ivan_kudinov48
3,Пермь,Пермский край,1 034 002,58.014965,56.246723,@ivan_kudinov48
4,Махачкала,Дагестан,623 254,42.983024,47.504872,@ivan_kudinov48


In [9]:
# Сохранение отфильтрованных данных в отдельный файл
output_csv = pc_csv + 'my_cities.csv'
filtered_data.to_csv(output_csv, index=False)

In [10]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"

In [11]:
# функция извлечения нужных погодных данных
def load(cityname:str, lat:float, lon:float):
    params = {
	"latitude": lat,
	"longitude": lon,
	"start_date": "1975-01-01",
	"end_date": "2025-05-31",
	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "snowfall", "snow_depth", "precipitation", "wind_speed_100m", "wind_direction_100m", "is_day"],
	"timezone": "Europe/Moscow",
	"temporal_resolution": "hourly_6"
    }
    responses = openmeteo.weather_api(url, params=params)

    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation {response.Elevation()} m asl")
    print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_rain = hourly.Variables(2).ValuesAsNumpy()
    hourly_snowfall = hourly.Variables(3).ValuesAsNumpy()
    hourly_snow_depth = hourly.Variables(4).ValuesAsNumpy()
    hourly_precipitation = hourly.Variables(5).ValuesAsNumpy()
    hourly_wind_speed_100m = hourly.Variables(6).ValuesAsNumpy()
    hourly_wind_direction_100m = hourly.Variables(7).ValuesAsNumpy()
    hourly_is_day = hourly.Variables(8).ValuesAsNumpy()

    hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
    )}

    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["rain"] = hourly_rain
    hourly_data["snowfall"] = hourly_snowfall
    hourly_data["snow_depth"] = hourly_snow_depth
    hourly_data["precipitation"] = hourly_precipitation
    hourly_data["wind_speed_100m"] = hourly_wind_speed_100m
    hourly_data["wind_direction_100m"] = hourly_wind_direction_100m
    hourly_data["is_day"] = hourly_is_day

    hourly_dataframe = pd.DataFrame(data = hourly_data)
    # print(hourly_dataframe)
    return hourly_dataframe  

iter_cnt = 0
iter_lim = 10 # задаем количество выгрузок данных за один запуск

# Предполагается, что filtered_data и load() уже определены
for index, row in filtered_data.iterrows():
    city, lat, lon = row['name'], row['latitude'], row['longitude']
    filename = f'{pc_csv}{city}.csv'
    exists = True
    try:
        frame = pd.read_csv(filename)
    except FileNotFoundError:
        exists = False
    except pd.errors.EmptyDataError:
        exists = False
    if not exists or frame.shape[1] == 0:
        iter_cnt += 1
        if iter_cnt > iter_lim:
            break
        print(f"Calling load with city={city}, lat={lat}, lon={lon}")
        frame = load(city, lat, lon)
        frame['city'] = city
        frame.to_csv(filename, index=False)
    elif 'city' not in frame.columns:
        frame['city'] = city
        frame.to_csv(filename, index=False)

Calling load with city=Москва, lat=55.625578, lon=37.6063916
Coordinates 55.641475677490234°N 37.60649108886719°E
Elevation 192.0 m asl
Timezone b'Europe/Moscow'b'GMT+3'
Timezone difference to GMT+0 10800 s
Calling load with city=Красноярск, lat=56.0091173, lon=92.872586
Coordinates 55.99296951293945°N 92.95082092285156°E
Elevation 162.0 m asl
Timezone b'Europe/Moscow'b'GMT+3'
Timezone difference to GMT+0 10800 s
Calling load with city=Краснодар, lat=45.0351532, lon=38.9772396
Coordinates 45.02635955810547°N 38.990684509277344°E
Elevation 31.0 m asl
Timezone b'Europe/Moscow'b'GMT+3'
Timezone difference to GMT+0 10800 s
Calling load with city=Пермь, lat=58.014965, lon=56.246723
Coordinates 58.03163146972656°N 56.27450942993164°E
Elevation 135.0 m asl
Timezone b'Europe/Moscow'b'GMT+3'
Timezone difference to GMT+0 10800 s
Calling load with city=Махачкала, lat=42.9830241, lon=47.5048717
Coordinates 42.98769760131836°N 47.473995208740234°E
Elevation 3.0 m asl
Timezone b'Europe/Moscow'b'GMT+

OpenMeteoRequestsError: {'reason': 'Minutely API request limit exceeded. Please try again in one minute.', 'error': True}